## Example: Using the ecFactory workflow engine for strain design

In [1]:
import sys
import cobra
from strainOptimizer import (strainOptimizer_engine,WorkflowParameters)

# Set tolerance
# cobra.Configuration().tolerance = 1e-9

### (Optional) Add heterologous pathway into model
If your target product is heterologous, you can add the heterologous pathway into the model. Please refer to the example [add_sclareol_pathway.py](0.add_sclareol_pathway.py)

### 1. Prepare parameters

StrainOptimizer provides a class `WorkflowParameters` to store the parameters for the workflow. It includes three types of parameters: `model`, `strain`, and `algorithm`.
- model: Model-related parameters
    - model_path: The path to the model file
    - model_type: The type of the model, e.g., 'ecGEM', 'etfl', 'gem', 'GAN_ec'
    - solver: The solver to be used
    - growth_id: The growth reaction id
- strain: Strain-related parameters
    - target_id: The target reaction id
    - product_name: The name of the product
    - c_source: The carbon source reaction id
    - c_uptake: The carbon source uptake rate
- algorithm: Algorithm-related parameters
    - design_algorithm: The design algorithm to be used. Currently, the supported algorithms are 'ecFactory', 'iBridge', 'ecFSEOF'.
    - simulation_method: The simulation method to be used. Currently, the supported methods are 'ppfba', 'pfba', 'moma', 'mopa'.
    - remove_essential: Whether to remove the essential reactions.
    - output_directory: The directory to save the results.
    - save_results: Whether to save the results.
    - experimental_yield: The experimental yield of the product.
    - scanning_range: The range of the scanning.
    

In [2]:
model_params = {
    'model_path': 'models/yeast/ecYeastGEM_batch.xml',
    'model_type': 'ecGEM',
    'solver': 'optlang-gurobi',
    'growth_id': 'r_2111',
}


# Strain parameters - target product and growth conditions
strain_params = {
    'target_id': 'r_1589',
    'product_name': '2-phenylethanol',
    'c_source': 'r_1714_REV',  # glucose exchange reaction
    'c_uptake': 1,  # glucose uptake rate (mmol/gDW/h)
}

# Algorithm control parameters - workflow and output settings
algorithm_params = {
    'design_algorithm': 'ecFactory',
    'simulation_method': 'ppfba',
    'experimental_yield': None, # if without experimental yield data, use the 1/2 
    'remove_essential': True,
    'output_directory': './results',
    'steps':123,
    'action_thresholds':[0.05,0.3,1.1],
    'save_results': True,
    'calculate_fcc': True,
    # 'only_final_result': True,
    # Note: ecFactory-specific parameters like steps, action_thresholds, etc.
    # would need to be added to AlgorithmControl if they're used

}

# Create WorkflowParameters using the three-level structure
params = WorkflowParameters(
    model=model_params,
    strain=strain_params,
    algorithm=algorithm_params
)

### 2. Run the ecFactory design workflow for ecModel

In [3]:
# Create workflow engine using the new framework
engine = strainOptimizer_engine(params)
print(f"Engine created for {params.strain['product_name']} cell factory design")
print(f"Target reaction: {params.strain['target_id']}")
print(f"Carbon source: {params.strain['c_source']}")
print(f"Model type: {params.model['model_type']}")
print(f"Algorithm: {params.algorithm['design_algorithm']}")

# Load model
model=engine.load_model()

Parameters validated successfully
Engine created for 2-phenylethanol cell factory design
Target reaction: r_1589
Carbon source: r_1714_REV
Model type: ecGEM
Algorithm: ecFactory
Loading ecGEM model from models/yeast/ecYeastGEM_batch.xml


In [4]:
# Run the design workflow
final_result = engine.run_design()
summary = engine.get_results_summary()
summary


Starting strain design workflow using ecFactory
Running ecFactory algorithm
Max yield: 0.4836632484699423
r_2111 rate: 0.08713483619135093
Scanning range for FSEOF: (0.09673264969398847, 0.4352969236229481)
 Experimental biomass yield set to: 0.266 g/gSubstrate
1.-  **** Running ecFSEOF ****
simulate WT-like: 0.0862634878294376
when growth rate is 0.0174,production rate is 0.433855785528911
when growth rate is 0.0215,production rate is 0.4086764593596936
when growth rate is 0.0256,production rate is 0.38349713319046397
when growth rate is 0.0296,production rate is 0.35893193692781006
when growth rate is 0.0337,production rate is 0.3337526107585851
when growth rate is 0.0378,production rate is 0.30857328458935973
when growth rate is 0.0418,production rate is 0.2840080883267001
when growth rate is 0.0459,production rate is 0.25882876215747924
when growth rate is 0.05,production rate is 0.23364943598825005
when growth rate is 0.054,production rate is 0.20908423972560128
when growth rate i

D:\code\github\strainOptimizer\src\strainOptimizer\strainDesign\ecFactory\ecfseof.py:231: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  FC["k_rxns"] = FC["k_rxns"][order]
D:\code\github\strainOptimizer\src\strainOptimizer\strainDesign\ecFactory\ecfseof.py:260: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  k_genes = k_genes[order]
D:\code\github\strainOptimizer\src\strainOptimizer\strainDesign\ecFactory\ecFactory_other.py:168: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the futur

ecFSEOF returned 25 targets
2.-  **** Find flux leak targets to block ****
2 leak rxn has been found and added to candidates.

3.-  **** Removing essential targets ****
Removing 0 essential targets

4.-  **** Construct Genes-metabolites network for classification of targets ****
independent targets: 16
target groups: 5
5.-  **** Running EUVA analysis ****
5.1-  **** Running EUVA for optimal production conditions ****
 Fix suboptimal experimental biomass = 0.05 h-1
5.2-  **** Running EUVA for optimal biomass growth condition ****
  - Maximize biomass production
5.3-  **** Discarding targets according to EUVA results ****
  - Discard OE targets with min=parsimonious=0: 5 targets removed
  - Discard KO targets with min>0: 0 targets removed
  - Discard KD and KO targets that belong to isoenzyme groups with optimal isoform for biomass formation: 0 targets removed
  - Discard targets with inconsistent result between EUVA and fseof: 0 targets removed
5.4-  ****  **** Rank targets by priority 

D:\code\github\strainOptimizer\src\strainOptimizer\strainDesign\ecFactory\ecFactory_other.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'up_distinct' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_gene_euvr_result[up_distinct_list]='up_distinct'
D:\code\github\strainOptimizer\src\strainOptimizer\strainDesign\ecFactory\run_ecFactory.py:316: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'no enzyme' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  genetable.loc[no_enzyme_list,'EUVA_priority_level']='no enzyme'


optimal production yield is:  0.23829259017662952 mmol/mmol carbon source
optimal production rate is:  0.23257764136800746 mmol/gDW/h
  - Minimal set of targets: 5
7.-  **** Calculating FCC scores for L1 genes ****
  - NGAM reaction: r_4046 (default)


C:\Users\wangh\AppData\Roaming\Python\Python313\site-packages\cobra\util\solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)
C:\Users\wangh\AppData\Roaming\Python\Python313\site-packages\cobra\util\solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)
C:\Users\wangh\AppData\Roaming\Python\Python313\site-packages\cobra\util\solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)
C:\Users\wangh\AppData\Roaming\Python\Python313\site-packages\cobra\util\solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)
C:\Users\wangh\AppData\Roaming\Python\Python313\site-packages\cobra\util\solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)
C:\Users\wangh\AppData\Roaming\Python\Python313\site-packages\cobra\util\so

  - FCC calculated: 22 genes, skipped (multi/no enzyme): 5
FCC filtered result: 9 genes retained
Results saved to results
Strain design workflow completed successfully


{'algorithm': 'ecFactory',
 'target_reaction': 'r_1589',
 'product': '2-phenylethanol',
 'status': 'completed',
 'num_targets': 27,
 'ko_targets': 2,
 'kd_targets': 5,
 'oe_targets': 20}

#### Parse result
- Overview of Combined Result DataFrame:
   - Index: gene IDs.
   - gene_name: Corresponding gene names.
   - k_score: ecFSEOF score (from Level 1 results).
   - action: Suggested genetic modification — OE (overexpression), KO (knockout), or KD (knockdown).
   - EUVA_priority_level: Priority level assigned by EUVA (Level 2 results).
   - minimal_candidates_set: Membership in the minimal candidate set (Level 3 results).



In [ ]:
final_result.head()

,gene_name,k_score,action,EUVA_priority_level,minimal_candidates_set
YOL086C,ADH1,1000.000000,OE,3.0,0.0
YDR380W,ARO10,1000.000000,OE,1.0,1.0
YDL168W,SFA1,1000.000000,OE,discarded by EUVA,0.0
YBR145W,ADH5,1000.000000,OE,discarded by EUVA,0.0
YGR256W,GND2,546.398499,OE,discarded by EUVA,0.0


In [ ]:
result_level1=engine.all_results['level 1 result']
result_level2=engine.all_results['level 2 result']
result_level3=engine.all_results['level 3 result']
result_fcc=engine.all_results.get('fcc_filtered_result', None)

### 3. Run the ecFactory design workflow for ETFL model

In [ ]:
efl_model_params = {
    'model_path': 'models\yeast\yeast8_cEFL_2584_enz_64_bins__20231221_083715.json',
    'model_type': 'etfl',
    # 'solver': 'optlang-gurobi',
    'solver': 'optlang-cplex',  # In our test, CPLEX is more stable for ETFL model simulation.
    'growth_id': 'r_2111',
    'c_source': 'r_1714',
    'c_uptake': 1,
}

engine.update_parameters(update_dict=efl_model_params)

print(f"Engine created for {params.strain['product_name']} cell factory design")
print(f"Target reaction: {params.strain['target_id']}")
print(f"Carbon source: {params.strain['c_source']}")
print(f"Model type: {params.model['model_type']}")
print(f"Algorithm: {params.algorithm['design_algorithm']}")

In [ ]:
# Run the design workflow
final_result = engine.run_design()
summary = engine.get_results_summary()
summary